# 02_Preprocessing.ipynb

## **0. Giới thiệu**

Notebook này thực hiện tiền xử lý dữ liệu theo đúng yêu cầu bài lab:

- Làm sạch và chuẩn hoá dữ liệu.

- Xử lý giá trị không hợp lệ.

- Xử lý missing values.

- Phát hiện và loại bỏ/điều chỉnh outlier (nếu cần).

- Chuẩn hoá dữ liệu (Normalization / Standardization).

- Feature engineering cơ bản.

## **1. Import thư viện và load dữ liệu từ file `01_data_exploration.ipynb`**

In [ ]:
import numpy as np
import sys, os

project_root = os.path.abspath("..")
sys.path.append(project_root)


from src.data_processing import *


data = np.load("../data/processed/data_raw.npy", allow_pickle=True)
headers = np.load("../data/processed/headers.npy", allow_pickle=True)

## **2. Xử lý giá trị không hợp lệ**

### Xử lý experience (<1, >20)

In [19]:
exp_col = data[:, headers.tolist().index("experience")].astype(str)
exp_clean = np.char.strip(exp_col)


exp_clean = np.where(exp_clean == '<1', '0', exp_clean)
exp_clean = np.where(exp_clean == '>20', '21', exp_clean)
exp_clean = np.where(exp_clean == '', 'nan', exp_clean)


exp_numeric = exp_clean.astype(float)
data[:, headers.tolist().index("experience")] = exp_clean

## **3. Xử lý Missing Values**

### 3.1 Numeric: fill bằng median

In [ ]:
numeric_cols = ["city_development_index", "experience", "training_hours"]

for c in numeric_cols:
    idx = headers.tolist().index(c)
    col = data[:, idx].astype(str)

    mask = missing_mask(col)
    col = np.where(mask, np.nan, col).astype(float)

    median = np.nanmedian(col)
    col[mask] = median

    data[:, idx] = col


### 3.2 Categorical: fill bằng mode

In [ ]:
categorical_cols = [h for h in headers if h not in numeric_cols and h != "target"]


for c in categorical_cols:
    idx = headers.tolist().index(c)
    col = data[:, idx].astype(str)
    data[:, idx] = fill_mode(col)

In [ ]:
for c in categorical_cols:
    idx = headers.tolist().index(c)
    col = data[:, idx].astype(str)
    data[:, idx] = label_encode(col)

## **4. Phát hiện & xử lý Outlier**

In [ ]:
for c in numeric_cols:
    idx = headers.tolist().index(c)
    arr = data[:, idx].astype(float)
    data[:, idx] = remove_outlier_iqr(arr)

## **5. Chuẩn hoá dữ liệu**

In [ ]:
for c in numeric_cols:
    idx = headers.tolist().index(c)
    arr = data[:, idx].astype(float)
    data[:, idx] = minmax(arr)

## **6. Standardization (Z-score)**

In [ ]:
# Ví dụ chuẩn hoá riêng training_hours
col_idx = headers.tolist().index("training_hours")
arr = data[:, col_idx].astype(float)
data[:, col_idx] = standardize(arr)

## **7. Feature Engineering**

In [26]:
cdi = data[:, headers.tolist().index("city_development_index")].astype(float)
train = data[:, headers.tolist().index("training_hours")].astype(float)


new_feature = (cdi * train).reshape(-1, 1)

data = np.hstack([data, new_feature])
headers = np.append(headers, "cdi_x_training")

## **8. Lưu dữ liệu đã tiền xử lý**

In [27]:
np.save("../data/processed/data_clean.npy", data)
np.save("../data/processed/headers_clean.npy", headers)

In [ ]:
# Xuất lại file CSV sạch cuối cùng

clean_csv_path = "../data/processed/data_clean.csv"

# Ghép header vào data
data_str = data.astype(str)
full_data = np.vstack([headers, data_str])

# Lưu ra CSV
np.savetxt(clean_csv_path, full_data, delimiter=",", fmt="%s")